In [1]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
import pandas as pd
# Load cấu hình
load_dotenv()
PG_HOST     = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT     = os.getenv("POSTGRES_PORT", "5432")
PG_DB       = os.getenv("POSTGRES_DB", "binance_dw")
PG_USER     = os.getenv("POSTGRES_USER", "binance")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD", "binance123")

DATABASE_URL = f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"

W = 75  # Output width
engine = create_engine(DATABASE_URL)
def fetch(query: str) -> pd.DataFrame:
    """Run an SQL query and return a DataFrame."""
    try:
        return pd.read_sql_query(query, DATABASE_URL)
    except Exception as e:
        print(f"  [ERROR] Query failed: {e}")
        return pd.DataFrame()

In [3]:
df = pd.read_sql("""
    SELECT
        COUNT(*) FILTER (WHERE is_anomaly IS NULL) AS null_count,
        MIN(trade_time) FILTER (WHERE is_anomaly IS NULL) AS null_earliest,
        MAX(trade_time) FILTER (WHERE is_anomaly IS NULL) AS null_latest
    FROM fact_binance_trades;
""", engine)
print(df)

   null_count null_earliest null_latest
0           0          None        None


In [8]:
df = pd.read_sql("""
    SELECT transaction_id, trade_time, price, quantity,
           is_anomaly,
           ROUND(z_score::numeric, 6) AS z_score,
           ROUND(price_dev_pct::numeric, 6) AS price_dev_pct
    FROM fact_binance_trades
    ORDER BY trade_time DESC
    LIMIT 20;
""", engine)
print(df.to_string())

        transaction_id              trade_time     price  quantity  is_anomaly  z_score  price_dev_pct
0   ETHUSDT_4128167590 2026-06-14 00:16:32.446   1674.46   0.59740       False      0.0            0.0
1   ETHUSDT_4128167592 2026-06-14 00:16:32.446   1674.46   0.38570       False      0.0            0.0
2   ETHUSDT_4128167589 2026-06-14 00:16:32.446   1674.46   0.01060       False      0.0            0.0
3   ETHUSDT_4128167591 2026-06-14 00:16:32.446   1674.46   0.00300       False      0.0            0.0
4   BTCUSDT_6406492933 2026-06-14 00:16:31.599  63954.84   0.01328       False      0.0            0.0
5   BTCUSDT_6406492932 2026-06-14 00:16:31.470  63954.85   0.00053       False      0.0            0.0
6   SOLUSDT_1980846424 2026-06-14 00:16:31.394     67.95   0.49400       False      0.0            0.0
7   BTCUSDT_6406492931 2026-06-14 00:16:31.186  63954.84   0.00421       False      0.0            0.0
8   ETHUSDT_4128167582 2026-06-14 00:16:31.167   1674.46   0.00620       

In [23]:
try:
    df = pd.read_sql("""
        SELECT 
            batch_id, sink_name, row_count, latency_ms,
            ROUND((row_count::numeric / NULLIF(latency_ms, 0)) * 1000, 2) AS throughput_per_sec,
            recorded_at
        FROM fact_pipeline_latency
        ORDER BY recorded_at DESC
        LIMIT 10;
    """, engine)
    print(df)
except Exception as e:
    print(f"Có lỗi xảy ra: {e}")

   batch_id   sink_name  row_count  latency_ms  throughput_per_sec  \
0     26382    BigQuery       6000        5809             1032.88   
1     26383  PostgreSQL       1173         125             9384.00   
2     26382  PostgreSQL       2000         148            13513.51   
3     26381  PostgreSQL       2000         128            15625.00   
4     26379    BigQuery       6000        6359              943.54   
5     26380  PostgreSQL       2000         167            11976.05   
6     26379  PostgreSQL       2000         126            15873.02   
7     26378  PostgreSQL       2000         175            11428.57   
8     26376    BigQuery       6000        6123              979.91   
9     26377  PostgreSQL       2000         185            10810.81   

                 recorded_at  
0 2026-06-13 16:34:16.902226  
1 2026-06-13 16:34:16.534584  
2 2026-06-13 16:34:11.088431  
3 2026-06-13 16:34:06.170139  
4 2026-06-13 16:34:02.441938  
5 2026-06-13 16:34:01.098448  
6 2026-06-13

In [ ]:
# Check 1000 bản ghi cuối cùng để phát hiện trùng lặp
try:
    df = pd.read_sql("""
        SELECT
            MIN(trade_time) AS earliest_trade_time,
            MAX(trade_time) AS latest_trade_time,
            COUNT(*) AS total_rows,
            COUNT(DISTINCT transaction_id) AS unique_ids
        FROM (
            SELECT transaction_id, trade_time
            FROM fact_binance_trades
            ORDER BY trade_time DESC
            LIMIT 1000
        ) AS sample;
    """, engine)
    print(df)

    df = pd.read_sql("""
        SELECT transaction_id, COUNT(*) AS occurrences
        FROM fact_binance_trades
        GROUP BY transaction_id
        HAVING COUNT(*) > 1
        ORDER BY occurrences DESC
        LIMIT 5;
    """, engine)

    if df.empty:
        print("  [✓] Không có bản ghi trùng lặp.")
    else:
        print(f"  [✗] CẢNH BÁO: PHÁT HIỆN {len(df):,} KHÓA TRÙNG LẶP (hiển thị tối đa 5)!")
        print("  Chi tiết các khóa bị trùng lặp:")
        for _, row in df.iterrows():
            print(f"    - Khóa trùng: {row['transaction_id']:<35} | Số lần xuất hiện: {row['occurrences']} lần")
except Exception as e:
    print(f"Có lỗi xảy ra: {e}")

      earliest_trade_time       latest_trade_time  total_rows  unique_ids
0 2026-06-13 23:27:21.385 2026-06-13 23:27:36.204        1000        1000
  [✓] Không có bản ghi trùng lặp.


In [28]:
overview_query = """
        SELECT 'fact_binance_trades' AS table_name, COUNT(*) AS row_count FROM fact_binance_trades
        UNION ALL SELECT 'fact_pipeline_latency',     COUNT(*) FROM fact_pipeline_latency
        UNION ALL SELECT 'dim_crypto_pair',           COUNT(*) FROM dim_crypto_pair
        UNION ALL SELECT 'dim_volume_category',       COUNT(*) FROM dim_volume_category
        UNION ALL SELECT 'dim_exchange_rate',         COUNT(*) FROM dim_exchange_rate
        UNION ALL SELECT 'dim_date',                  COUNT(*) FROM dim_date
        UNION ALL SELECT 'dim_time',                  COUNT(*) FROM dim_time
        ORDER BY row_count DESC;
    """
    df_overview = fetch(overview_query)
    if not df_overview.empty:
        df_overview['row_count'] = df_overview['row_count'].apply(fmt_int)
        print_df(df_overview)

    # Total volume
    df_vol = fetch("SELECT SUM(amount_usd) AS total, MIN(trade_time) AS first_tx, MAX(trade_time) AS last_tx FROM fact_binance_trades;")
    if not df_vol.empty and pd.notnull(df_vol['total'].iloc[0]):
        print(f"\n  Total volume      : {fmt_money(df_vol['total'].iloc[0])}")
        print(f"  First transaction : {df_vol['first_tx'].iloc[0]}")
        print(f"  Last transaction  : {df_vol['last_tx'].iloc[0]}")


IndentationError: unexpected indent (3904967815.py, line 11)